# Dataset3 — YOLO26s baseline @ 768

Notebook này huấn luyện baseline thuần `yolo26s.pt` trên Dataset3. Không dùng SCConv, P2, iAFF, MultiSEAM, DySample, hard mining, weighted sampling, class weighting hoặc custom loss.

Validation chỉ dùng Vietnam `valid`; test split không được đánh giá trong notebook này. Các điều kiện an toàn đều dừng bằng lỗi rõ ràng thay vì tự thay đổi protocol.

In [ ]:
%pip install -q ultralytics==8.4.155

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

ULTRALYTICS_VERSION = "8.4.155"
MODEL_NAME = "yolo26s.pt"
IMG_SIZE = 768
EPOCHS = 100
BATCH_SIZE = 16
SEED = 0
DETERMINISTIC = True
OPTIMIZER = "auto"
VALIDATION_SPLIT = "val"
CLASS_NAMES = [
    "motorcycle",
    "rider_with_helmet",
    "rider_without_helmet",
]
NO_HELMET = "rider_without_helmet"
EXPECTED_IMAGE_COUNTS = {"train": 4330, "valid": 140, "test": 100}
FORBIDDEN_MODULE_NAMES = {
    "C3k2_SCConv", "ScConv", "SRU", "CRU", "P2",
    "iAFF", "MultiSEAM", "DySample", "CBAM", "ECA",
    "AFPN", "ADown",
}

assert ultralytics.__version__ == ULTRALYTICS_VERSION, (
    f"Expected ultralytics=={ULTRALYTICS_VERSION}, got {ultralytics.__version__}"
)
assert torch.cuda.is_available(), "This protocol requires a CUDA GPU as device=0."
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

In [ ]:
DRIVE_ROOT = Path("/content/drive")
DATASET_NAME = "Dataset3"
DATASET_ROOT_CANDIDATES = []
for mount_root in [DRIVE_ROOT / "MyDrive", DRIVE_ROOT / "Shareddrives"]:
    if not mount_root.exists():
        continue
    DATASET_ROOT_CANDIDATES.append(mount_root / DATASET_NAME)
    try:
        mount_children = list(mount_root.iterdir())
    except OSError as error:
        raise RuntimeError(
            f"Cannot inspect mounted Drive root: {mount_root}. "
            "Remount Google Drive and rerun this cell."
        ) from error
    for child in mount_children:
        DATASET_ROOT_CANDIDATES.append(child / DATASET_NAME)

DATA_YAML_CANDIDATES = [
    root / "data.yaml"
    for root in DATASET_ROOT_CANDIDATES
]
DATA_YAML = next(
    (candidate for candidate in DATA_YAML_CANDIDATES if candidate.exists()),
    None,
)
if DATA_YAML is None:
    checked = [str(path) for path in DATA_YAML_CANDIDATES]
    raise FileNotFoundError(
        "Dataset3/data.yaml was not found. Checked: "
        + str(checked)
        + ". Share Dataset3 with this account or create a shortcut named Dataset3 in My Drive."
    )
EXPERIMENT_NAME = "dataset3_yolo26s_baseline_768"
DATASET_ROOT = DATA_YAML.parent
OUTPUT_ROOT_CANDIDATES = []
for mount_root in [DRIVE_ROOT / "MyDrive", DRIVE_ROOT / "Shareddrives"]:
    if not mount_root.exists():
        continue
    OUTPUT_ROOT_CANDIDATES.append(mount_root / "DetectedHelmets")
    try:
        output_children = list(mount_root.iterdir())
    except OSError as error:
        raise RuntimeError(
            f"Cannot inspect mounted Drive root: {mount_root}. "
            "Remount Google Drive and rerun this cell."
        ) from error
    for child in output_children:
        OUTPUT_ROOT_CANDIDATES.append(child / "DetectedHelmets")

resume_output_root = next(
    (
        candidate
        for candidate in OUTPUT_ROOT_CANDIDATES
        if (candidate / "runs" / "train" / EXPERIMENT_NAME / "weights" / "last.pt").exists()
    ),
    None,
)
existing_output_root = next(
    (candidate for candidate in OUTPUT_ROOT_CANDIDATES if candidate.exists()),
    None,
)
OUTPUT_ROOT = (
    resume_output_root
    or existing_output_root
    or Path("/content/drive/MyDrive/DetectedHelmets")
)
if OUTPUT_ROOT.name != "DetectedHelmets":
    raise RuntimeError(
        f"Invalid output root: {OUTPUT_ROOT}. "
        "Create a DetectedHelmets shortcut in My Drive and remount Drive."
    )
RUNS_DIR = OUTPUT_ROOT / "runs" / "train"
EXPERIMENT_DIR = RUNS_DIR / EXPERIMENT_NAME

print(f"Using Dataset YAML: {DATA_YAML}")
print(f"Using output root: {OUTPUT_ROOT}")

with DATA_YAML.open("r", encoding="utf-8") as handle:
    data_config = yaml.safe_load(handle)

assert isinstance(data_config, dict), "data.yaml must contain a mapping."
assert int(data_config.get("nc", -1)) == 3, f"Expected nc=3, got {data_config.get('nc')}"

raw_names = data_config.get("names")
if isinstance(raw_names, dict):
    names = [raw_names[key] for key in sorted(raw_names, key=lambda value: int(value))]
else:
    names = list(raw_names or [])
assert names == CLASS_NAMES, f"Unexpected class names: {names}"

def yaml_split_value(config, keys):
    for key in keys:
        if key in config and config[key] is not None:
            return config[key]
    raise KeyError(f"Missing one of dataset split keys: {keys}")

def resolve_images_dir(raw_value):
    raw_path = Path(str(raw_value))
    yaml_path_root = data_config.get("path")
    candidates = []
    if raw_path.is_absolute():
        candidates.append(raw_path)
    else:
        if yaml_path_root:
            yaml_root = Path(str(yaml_path_root))
            if not yaml_root.is_absolute():
                yaml_root = DATA_YAML.parent / yaml_root
            candidates.append(yaml_root / raw_path)
        candidates.extend([DATA_YAML.parent / raw_path, DATASET_ROOT / raw_path])
    for candidate in candidates:
        if candidate.name.lower() == "images" and candidate.exists():
            return candidate
        if (candidate / "images").exists():
            return candidate / "images"
    candidate = candidates[0]
    return candidate if candidate.name.lower() == "images" else candidate / "images"

SPLIT_DIRS = {
    "train": resolve_images_dir(yaml_split_value(data_config, ["train"])),
    "valid": resolve_images_dir(yaml_split_value(data_config, ["valid", "val"])),
    "test": resolve_images_dir(yaml_split_value(data_config, ["test"])),
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def image_files(images_dir):
    return sorted(
        path for path in images_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )

actual_counts = {split: len(image_files(path)) for split, path in SPLIT_DIRS.items()}
print("Resolved split directories:")
for split, path in SPLIT_DIRS.items():
    print(f"  {split}: {path}")
print("Actual image counts:", actual_counts)
if actual_counts != EXPECTED_IMAGE_COUNTS:
    print("WARNING: actual image counts differ from the required Dataset3 counts.")
    raise RuntimeError(f"Expected {EXPECTED_IMAGE_COUNTS}, got {actual_counts}. Stop for inspection.")

for split, images_dir in SPLIT_DIRS.items():
    labels_dir = images_dir.parent / "labels"
    images = image_files(images_dir)
    try:
        # List labels once; avoid one Drive FUSE stat call per image.
        label_files = list(labels_dir.glob("*.txt"))
    except ConnectionAbortedError as error:
        raise RuntimeError(
            f"Google Drive connection aborted while listing labels: {labels_dir}. "
            "Remount Drive and rerun this validation cell."
        ) from error
    label_stems = {label.stem for label in label_files}
    missing_labels = [image for image in images if image.stem not in label_stems]
    print(f"{split.title()} images: {len(images)}")
    print(f"{split.title()} labels: {len(label_files)}")
    if missing_labels:
        preview = [str(path) for path in missing_labels[:10]]
        raise FileNotFoundError(f"Missing image/label pairs in {split}: {preview}")

print("Dataset3 validation passed: YAML, split counts, and image/label pairs are valid.")

In [ ]:
model = YOLO(MODEL_NAME)
print("Loaded model:", MODEL_NAME)
model.info(verbose=True, imgsz=IMG_SIZE)

def module_class_names(yolo_model):
    return {type(module).__name__ for module in yolo_model.model.modules()}

module_names = module_class_names(model)
unexpected_custom_modules = sorted(module_names & FORBIDDEN_MODULE_NAMES)
assert not unexpected_custom_modules, (
    f"Baseline model contains forbidden custom modules: {unexpected_custom_modules}"
)
print("Baseline module audit passed.")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

device = torch.device("cuda:0")

model.model.to(device)
model.model.eval()

try:
    with torch.inference_mode():
        dummy = torch.zeros(
            BATCH_SIZE,
            3,
            IMG_SIZE,
            IMG_SIZE,
            device=device,
        )

        dummy_output = model.model(dummy)

    del dummy_output
    del dummy

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

except RuntimeError as error:
    if "out of memory" in str(error).lower():
        torch.cuda.empty_cache()

        raise RuntimeError(
            "CUDA OOM during 768px preflight with "
            f"batch={BATCH_SIZE}. "
            "Stop experiment; do NOT reduce batch."
        ) from error

    raise

# Return the model to CPU before Ultralytics creates its trainer.
# Otherwise the trainer can mix CUDA and CPU tensors while remapping the state dict.
model.model.to("cpu")
torch.cuda.empty_cache()

print(
    f"CUDA preflight PASS: "
    f"imgsz={IMG_SIZE}, batch={BATCH_SIZE}"
)

In [ ]:
LAST_PT = EXPERIMENT_DIR / "weights" / "last.pt"

if LAST_PT.exists():
    TRAIN_MODE = "resume"
    print(f"Resuming interrupted training from: {LAST_PT}")
    train_model = YOLO(str(LAST_PT))
    train_results = train_model.train(
        resume=True,
        data=str(DATA_YAML),
        imgsz=768,
        batch=16,
        device=0,
        workers=4,
    )
else:
    if EXPERIMENT_DIR.exists():
        raise FileNotFoundError(
            f"Experiment directory exists but last.pt is missing: {EXPERIMENT_DIR}. "
            "Restore last.pt or move this incomplete directory before starting fresh."
        )
    TRAIN_MODE = "fresh"
    print("Starting fresh training run.")
    train_model = model
    train_results = train_model.train(
        data=str(DATA_YAML),
        epochs=100,
        imgsz=768,
        batch=16,
        device=0,
        workers=4,
        optimizer="auto",
        seed=0,
        deterministic=True,
        patience=25,
        pretrained=True,
        amp=True,
        close_mosaic=10,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        cutmix=0.0,
        copy_paste=0.0,
        project=str(RUNS_DIR),
        name=EXPERIMENT_NAME,
        save=True,
        save_period=10,
        plots=True,
        exist_ok=False,
    )

trainer = getattr(train_model, "trainer", None)
actual_batch = getattr(trainer, "batch_size", None)
if actual_batch is None and trainer is not None:
    actual_batch = getattr(getattr(trainer, "args", None), "batch", None)
assert int(actual_batch) == BATCH_SIZE, f"Trainer batch changed: {actual_batch}"
print(f"Training completed with actual batch={actual_batch}.")

In [ ]:
BEST_PT = EXPERIMENT_DIR / "weights" / "best.pt"
assert BEST_PT.exists(), f"Best checkpoint not found: {BEST_PT}"
best_model = YOLO(str(BEST_PT))
best_module_names = module_class_names(best_model)
best_forbidden = sorted(best_module_names & FORBIDDEN_MODULE_NAMES)
assert not best_forbidden, f"Best model contains forbidden modules: {best_forbidden}"

validation_results = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=768,
    batch=16,
    device=0,
    plots=True,
)
TEST_SET_USED = False

def safe_f1(precision, recall):
    denominator = precision + recall
    return 2.0 * precision * recall / denominator if denominator else 0.0

def extract_validation_metrics(result):
    overall_precision = float(result.box.mp)
    overall_recall = float(result.box.mr)
    summary = {
        "Precision": overall_precision,
        "Recall": overall_recall,
        "F1": safe_f1(overall_precision, overall_recall),
        "mAP50": float(result.box.map50),
        "mAP50-95": float(result.box.map),
        "by_class": {},
    }
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision, recall, ap50, ap5095 = result.box.class_result(class_id)
        precision = float(precision)
        recall = float(recall)
        summary["by_class"][class_name] = {
            "Precision": precision,
            "Recall": recall,
            "F1": safe_f1(precision, recall),
            "AP50": float(ap50),
            "AP50-95": float(ap5095),
        }
    return summary

validation_summary = extract_validation_metrics(validation_results)
overall_table = pd.DataFrame([
    {"Metric": key, "Value": value}
    for key, value in validation_summary.items()
    if key != "by_class"
])
class_table = pd.DataFrame(validation_summary["by_class"]).T
display(overall_table)
display(class_table)

print("===== RIDER WITHOUT HELMET =====")
display(pd.DataFrame([validation_summary["by_class"][NO_HELMET]]))


In [ ]:
def confusion_matrix_array(result):
    matrix = getattr(getattr(result, "confusion_matrix", None), "matrix", None)
    matrix = np.asarray(matrix) if matrix is not None else None
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    assert matrix is not None and matrix.shape == expected_shape, (
        f"Expected confusion matrix shape {expected_shape}, got {None if matrix is None else matrix.shape}"
    )
    return matrix

confusion_matrix = confusion_matrix_array(validation_results)
confusion_labels = CLASS_NAMES + ["background"]
confusion_index = {label: index for index, label in enumerate(confusion_labels)}
nohelmet_index = confusion_index[NO_HELMET]
motorcycle_index = confusion_index["motorcycle"]
withhelmet_index = confusion_index["rider_with_helmet"]
background_index = confusion_index["background"]

# Ultralytics confusion matrix is reported here with the requested orientation: matrix[predicted, actual].
nohelmet_actual_total = confusion_matrix[:, nohelmet_index].sum()
confusion_stats = {
    "NoHelmet -> NoHelmet": float(confusion_matrix[nohelmet_index, nohelmet_index]),
    "NoHelmet -> Motorcycle": float(confusion_matrix[motorcycle_index, nohelmet_index]),
    "NoHelmet -> WithHelmet": float(confusion_matrix[withhelmet_index, nohelmet_index]),
    "NoHelmet -> Background": float(confusion_matrix[background_index, nohelmet_index]),
    "Background -> NoHelmet": float(confusion_matrix[nohelmet_index, background_index]),
    "NoHelmet diagonal": (
        float(confusion_matrix[nohelmet_index, nohelmet_index] / nohelmet_actual_total)
        if nohelmet_actual_total else 0.0
    ),
}
display(pd.DataFrame({"Statistic": list(confusion_stats), "Value": list(confusion_stats.values())}))

def find_artifact(filename):
    roots = [EXPERIMENT_DIR]
    validation_dir = Path(str(getattr(validation_results, "save_dir", "")))
    if validation_dir.exists():
        roots.append(validation_dir)
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob(filename))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

plot_specs = [
    ("results.png", "Training results"),
    ("BoxPR_curve.png", "Box PR curve"),
    ("BoxF1_curve.png", "Box F1 curve"),
    ("BoxP_curve.png", "Box precision curve"),
    ("BoxR_curve.png", "Box recall curve"),
    ("confusion_matrix.png", "Confusion matrix"),
    ("confusion_matrix_normalized.png", "Normalized confusion matrix"),
]
for filename, title in plot_specs:
    artifact = find_artifact(filename)
    if artifact is None:
        print(f"WARNING: artifact not found: {filename}")
        continue
    print(f"Displaying {title}: {artifact}")
    display(Image.open(artifact))

In [ ]:
RESULTS_CSV = EXPERIMENT_DIR / "results.csv"
assert RESULTS_CSV.exists(), f"Training results.csv not found: {RESULTS_CSV}"

results_df = pd.read_csv(RESULTS_CSV)
results_df.columns = [
    column.strip()
    for column in results_df.columns
]

required_columns = [
    "epoch",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]
missing = [
    column
    for column in required_columns
    if column not in results_df.columns
]
if missing:
    raise KeyError(f"Missing results.csv columns: {missing}")

print("First results.csv epochs:", results_df["epoch"].head().tolist())
first_raw_epoch = int(results_df["epoch"].iloc[0])
if first_raw_epoch not in (0, 1):
    raise ValueError(
        f"Unexpected results.csv epoch convention: first epoch={first_raw_epoch}"
    )
epoch_is_zero_based = first_raw_epoch == 0
print(
    "Epoch convention:",
    "0-based; report display epoch = raw epoch + 1"
    if epoch_is_zero_based
    else "1-based; report raw epoch"
)

def display_epoch(raw_epoch):
    raw_epoch = int(raw_epoch)
    return raw_epoch + 1 if epoch_is_zero_based else raw_epoch

best_map50_idx = results_df["metrics/mAP50(B)"].idxmax()
best_map5095_idx = results_df["metrics/mAP50-95(B)"].idxmax()
best_recall_idx = results_df["metrics/recall(B)"].idxmax()

best_map50_epoch = display_epoch(
    results_df.loc[best_map50_idx, "epoch"]
)
best_map5095_epoch = display_epoch(
    results_df.loc[best_map5095_idx, "epoch"]
)
best_recall_epoch = display_epoch(
    results_df.loc[best_recall_idx, "epoch"]
)

metric_epoch_rows = pd.DataFrame([
    {
        "Best metric": "mAP50",
        "Epoch": best_map50_epoch,
        "Value": results_df.loc[
            best_map50_idx,
            "metrics/mAP50(B)",
        ],
    },
    {
        "Best metric": "mAP50-95",
        "Epoch": best_map5095_epoch,
        "Value": results_df.loc[
            best_map5095_idx,
            "metrics/mAP50-95(B)",
        ],
    },
    {
        "Best metric": "Recall",
        "Epoch": best_recall_epoch,
        "Value": results_df.loc[
            best_recall_idx,
            "metrics/recall(B)",
        ],
    },
])
print("Evaluated checkpoint: best.pt")
display(metric_epoch_rows)

loss_pairs = {}
for loss_name, train_candidates, val_candidates in [
    ("box_loss", ("train/box_loss",), ("val/box_loss",)),
    ("cls_loss", ("train/cls_loss",), ("val/cls_loss",)),
    ("l1_loss", ("train/l1_loss", "train/dfl_loss"), ("val/l1_loss", "val/dfl_loss")),
]:
    train_column = next(
        (name for name in train_candidates if name in results_df.columns),
        None,
    )
    val_column = next(
        (name for name in val_candidates if name in results_df.columns),
        None,
    )
    if train_column and val_column:
        loss_pairs[loss_name] = (train_column, val_column)

assert loss_pairs, "No train/validation loss pairs found in results.csv."
loss_rows = []
for loss_name, (train_column, val_column) in loss_pairs.items():
    loss_rows.append({
        "Loss": loss_name,
        "Train column": train_column,
        "Validation column": val_column,
        "Final train": results_df[train_column].iloc[-1],
        "Final validation": results_df[val_column].iloc[-1],
    })
loss_table = pd.DataFrame(loss_rows)
display(loss_table)

def slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return 0.0
    return float(np.polyfit(np.arange(len(values)), values, 1)[0])

window = max(5, len(results_df) // 5)
overfit_signals = []
for loss_name, (train_column, val_column) in loss_pairs.items():
    train_values = results_df[train_column].to_numpy(dtype=float)
    val_values = results_df[val_column].to_numpy(dtype=float)
    early_gap = float(np.mean(val_values[:window] - train_values[:window]))
    late_gap = float(np.mean(val_values[-window:] - train_values[-window:]))
    validation_slope = slope(val_values[-window:])
    overfit_signals.append(
        late_gap > early_gap * 1.25
        and validation_slope > 0
    )

if any(overfit_signals):
    overfitting_assessment = "possible overfit"
elif all(
    slope(results_df[val_column].to_numpy(dtype=float)[-window:]) < 0
    for _, val_column in loss_pairs.values()
):
    overfitting_assessment = "possible underfit"
else:
    overfitting_assessment = "stable"

print("Overfitting check:", overfitting_assessment)
print("Assessment is based on train/validation loss trends, not training loss alone.")

In [ ]:
def extract_gflops(info):
    if isinstance(info, dict):
        for key in ("GFLOPs", "gflops", "flops"):
            if key in info:
                try:
                    return float(info[key])
                except (TypeError, ValueError):
                    pass
    if isinstance(info, (tuple, list)):
        for value in reversed(info):
            try:
                return float(value)
            except (TypeError, ValueError):
                pass
    return float("nan")

parameters = int(
    sum(parameter.numel() for parameter in best_model.model.parameters())
)
info = best_model.info(verbose=True, imgsz=IMG_SIZE)
gflops = extract_gflops(info)

validation_images = image_files(SPLIT_DIRS["valid"])
benchmark_images = validation_images[:min(50, len(validation_images))]
if len(benchmark_images) == 0:
    raise RuntimeError(
        "No validation images available for benchmark."
    )

best_model.model.to("cuda:0")
best_model.model.eval()

torch.cuda.empty_cache()

with torch.inference_mode():
    warmup_result = best_model.predict(
        source=str(benchmark_images[0]),
        imgsz=IMG_SIZE,
        device=0,
        batch=1,
        save=False,
        verbose=False,
    )

del warmup_result
torch.cuda.synchronize()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with torch.inference_mode():
    predictions = best_model.predict(
        source=[str(path) for path in benchmark_images],
        imgsz=IMG_SIZE,
        device=0,
        batch=1,
        save=False,
        verbose=False,
        stream=False,
    )

torch.cuda.synchronize()

inference_times = []
for result in predictions:
    if (
        not hasattr(result, "speed")
        or "inference" not in result.speed
    ):
        raise RuntimeError(
            "Ultralytics result does not contain speed['inference']."
        )
    inference_times.append(float(result.speed["inference"]))

if not inference_times:
    raise RuntimeError("No inference timing values collected.")

inference_ms = float(np.mean(inference_times))
inference_std_ms = float(np.std(inference_times))
inference_median_ms = float(np.median(inference_times))
fps = (
    1000.0 / inference_ms
    if inference_ms > 0
    else float("nan")
)
peak_vram_gb = float(
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
)

print(f"Benchmark images: {len(inference_times)}")
print(f"Inference mean: {inference_ms:.3f} ms/image")
print(f"Inference median: {inference_median_ms:.3f} ms/image")
print(f"Inference std: {inference_std_ms:.3f} ms")
print(f"Inference FPS: {fps:.2f}")
print(f"Peak allocated VRAM: {peak_vram_gb:.3f} GB")

del predictions
torch.cuda.empty_cache()

In [ ]:
nohelmet_metrics = validation_summary["by_class"][NO_HELMET]
motorcycle_metrics = validation_summary["by_class"]["motorcycle"]
withhelmet_metrics = validation_summary["by_class"]["rider_with_helmet"]

summary_row = {
    "Model": "YOLO26s",
    "Dataset": "Dataset3",
    "imgsz": IMG_SIZE,
    "epochs_requested": EPOCHS,
    "Evaluated_Checkpoint": "best.pt",
    "Best_mAP50_Epoch": best_map50_epoch,
    "Best_mAP50_95_Epoch": best_map5095_epoch,
    "Best_Recall_Epoch": best_recall_epoch,
    "Precision": validation_summary["Precision"],
    "Recall": validation_summary["Recall"],
    "F1": validation_summary["F1"],
    "mAP50": validation_summary["mAP50"],
    "mAP50-95": validation_summary["mAP50-95"],
    "Motorcycle_AP50": motorcycle_metrics["AP50"],
    "WithHelmet_AP50": withhelmet_metrics["AP50"],
    "NoHelmet_Precision": nohelmet_metrics["Precision"],
    "NoHelmet_Recall": nohelmet_metrics["Recall"],
    "NoHelmet_F1": nohelmet_metrics["F1"],
    "NoHelmet_AP50": nohelmet_metrics["AP50"],
    "NoHelmet_AP50-95": nohelmet_metrics["AP50-95"],
    "NoHelmet_diagonal": confusion_stats["NoHelmet diagonal"],
    "NoHelmet_to_WithHelmet": confusion_stats["NoHelmet -> WithHelmet"],
    "NoHelmet_to_Background": confusion_stats["NoHelmet -> Background"],
    "Background_to_NoHelmet": confusion_stats["Background -> NoHelmet"],
    "Parameters": parameters,
    "GFLOPs": gflops,
    "Inference_ms": inference_ms,
    "Inference_median_ms": inference_median_ms,
    "Inference_std_ms": inference_std_ms,
    "Inference_FPS": fps,
    "Inference_Peak_VRAM_GB": peak_vram_gb,
}
summary_df = pd.DataFrame([summary_row])
SUMMARY_CSV = EXPERIMENT_DIR / "baseline_dataset3_summary.csv"
SUMMARY_TXT = EXPERIMENT_DIR / "baseline_dataset3_summary.txt"
summary_df.to_csv(SUMMARY_CSV, index=False)

text_lines = [
    "DATASET3 YOLO26s BASELINE SUMMARY",
    f"Dataset3 train images: {actual_counts['train']}",
    f"Vietnam validation images: {actual_counts['valid']}",
    f"Vietnam test images: {actual_counts['test']}",
    f"Model: YOLO26s ({MODEL_NAME})",
    f"Input size: {IMG_SIZE}",
    f"Epochs requested: {EPOCHS}",
    "Evaluated checkpoint: best.pt",
    f"Best mAP50 epoch: {best_map50_epoch}",
    f"Best mAP50-95 epoch: {best_map5095_epoch}",
    f"Best Recall epoch: {best_recall_epoch}",
    f"Precision: {validation_summary['Precision']:.6f}",
    f"Recall: {validation_summary['Recall']:.6f}",
    f"F1: {validation_summary['F1']:.6f}",
    f"mAP50: {validation_summary['mAP50']:.6f}",
    f"mAP50-95: {validation_summary['mAP50-95']:.6f}",
    f"NoHelmet Precision: {nohelmet_metrics['Precision']:.6f}",
    f"NoHelmet Recall: {nohelmet_metrics['Recall']:.6f}",
    f"NoHelmet F1: {nohelmet_metrics['F1']:.6f}",
    f"NoHelmet AP50: {nohelmet_metrics['AP50']:.6f}",
    f"NoHelmet AP50-95: {nohelmet_metrics['AP50-95']:.6f}",
    f"NoHelmet normalized diagonal: {confusion_stats['NoHelmet diagonal']:.6f}",
    f"NoHelmet -> WithHelmet: {confusion_stats['NoHelmet -> WithHelmet']:.6f}",
    f"NoHelmet -> Background: {confusion_stats['NoHelmet -> Background']:.6f}",
    f"Background -> NoHelmet: {confusion_stats['Background -> NoHelmet']:.6f}",
    f"Parameters: {parameters}",
    f"GFLOPs: {gflops}",
    f"Inference mean ms/image: {inference_ms:.6f}",
    f"Inference median ms/image: {inference_median_ms:.6f}",
    f"Inference std ms: {inference_std_ms:.6f}",
    f"Inference FPS: {fps:.6f}",
    f"Peak allocated VRAM GB: {peak_vram_gb:.6f}",
    f"Overfitting check: {overfitting_assessment}",
    "Test split was not used.",
]
SUMMARY_TXT.write_text("\n".join(text_lines) + "\n", encoding="utf-8")
display(summary_df)
print("Saved:", SUMMARY_CSV)
print("Saved:", SUMMARY_TXT)

In [ ]:
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert MODEL_NAME == "yolo26s.pt"
assert DATASET_ROOT.name == "Dataset3"
assert IMG_SIZE == 768
assert EPOCHS == 100
assert BATCH_SIZE == 16
assert SEED == 0
assert DETERMINISTIC is True
assert OPTIMIZER == "auto"
assert not (module_class_names(model) & FORBIDDEN_MODULE_NAMES)
assert not (module_class_names(best_model) & FORBIDDEN_MODULE_NAMES)
assert SPLIT_DIRS["valid"].parent.parent == DATASET_ROOT
assert VALIDATION_SPLIT == "val"
assert TEST_SET_USED is False
assert SUMMARY_CSV.exists() and SUMMARY_TXT.exists()

print("========================================")
print("DATASET3 BASELINE TRAINING COMPLETE")
print("========================================")
print("Model:")
print("YOLO26s")
print("Dataset:")
print("Dataset3")
print("Input:")
print(IMG_SIZE)
print("Train images:")
print(actual_counts["train"])
print("Validation images:")
print(actual_counts["valid"])
print("Epoch requested:")
print(EPOCHS)
print("Evaluated checkpoint:")
print("best.pt")
print("Best mAP50 epoch:")
print(best_map50_epoch)
print("Best mAP50-95 epoch:")
print(best_map5095_epoch)
print("Best Recall epoch:")
print(best_recall_epoch)
print("Precision:")
print(validation_summary["Precision"])
print("Recall:")
print(validation_summary["Recall"])
print("F1:")
print(validation_summary["F1"])
print("mAP50:")
print(validation_summary["mAP50"])
print("mAP50-95:")
print(validation_summary["mAP50-95"])
print("RIDER_WITHOUT_HELMET")
print("Precision:")
print(nohelmet_metrics["Precision"])
print("Recall:")
print(nohelmet_metrics["Recall"])
print("F1:")
print(nohelmet_metrics["F1"])
print("AP50:")
print(nohelmet_metrics["AP50"])
print("AP50-95:")
print(nohelmet_metrics["AP50-95"])
print("Normalized confusion diagonal:")
print(confusion_stats["NoHelmet diagonal"])
print("NoHelmet -> WithHelmet:")
print(confusion_stats["NoHelmet -> WithHelmet"])
print("NoHelmet -> Background:")
print(confusion_stats["NoHelmet -> Background"])
print("Background -> NoHelmet:")
print(confusion_stats["Background -> NoHelmet"])
print("TEST SET USED:")
print("NO")
print("Comparison targets only (not hard-coded success criteria):")
print("  NoHelmet Precision >= 0.90")
print("  NoHelmet Recall >= 0.90")
print("  NoHelmet F1 >= 0.90")
print("  NoHelmet AP50 >= 0.90")
print("  NoHelmet normalized diagonal >= 0.90")
print("  Overall mAP50 >= 0.92")
print("========================================")